🏆 Detección de Fake News: Regresión Logística Optimizada
Este notebook contiene la versión final que nos dio el mejor resultado en Kaggle (Score: 0.616).
El proceso consta de tres pasos:

Carga y enriquecimiento de texto (Contexto).

Búsqueda de parámetros con Optuna.

Entrenamiento final con los parámetros ganadores fijados.

In [1]:
!pip install optuna -q

import pandas as pd
import optuna
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
import warnings

warnings.filterwarnings('ignore')

# 1. FUNCIÓN PARA AÑADIR CONTEXTO POLÍTICO
def preprocesar_y_enriquecer(df):
    df_limpio = df.copy()
    
    # Rellenar nulos
    for col in ['speaker_job', 'state_info', 'speaker', 'party_affiliation', 'subject']:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna('unknown')
            
    # Limpiar texto de la frase
    if 'statement' in df_limpio.columns:
        df_limpio['statement_clean'] = df_limpio['statement'].str.lower().str.replace(r'[^a-z0-9\s-]', '', regex=True)
        
    # Unir todo en una sola columna
    df_limpio['texto_final'] = (
        "speaker " + df_limpio['speaker'].astype(str) + 
        " party " + df_limpio['party_affiliation'].astype(str) + 
        " subject " + df_limpio['subject'].astype(str) + 
        " said " + df_limpio['statement_clean'].astype(str)
    )
    return df_limpio

# 2. CARGA DE DATOS
print("Cargando y preparando datos con contexto...")
train = pd.read_csv("../../data/train.csv")
test = pd.read_csv("../../data/test_nolabel.csv")

train_clean = preprocesar_y_enriquecer(train)
test_clean = preprocesar_y_enriquecer(test)

X_train = train_clean['texto_final'] 
y_train = train_clean['label']
X_test = test_clean['texto_final']

print("¡Datos listos para Optuna!")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando y preparando datos con contexto...
¡Datos listos para Optuna!


🔍 Búsqueda de Hiperparámetros con Optuna
En esta sección exploramos combinaciones con un límite de seguridad (C >= 0.1). Aunque Optuna busque, en la siguiente celda forzaremos el entrenamiento con los parámetros que ya sabemos que nos dan la victoria.

In [2]:
def objective_contexto(trial):
    # 1. TF-IDF Palabras (dejamos que Optuna explore vocabularios más grandes)
    w_min_df = trial.suggest_int('word_min_df', 1, 4)
    w_max_df = trial.suggest_float('word_max_df', 0.7, 1.0)
    w_max_feat = trial.suggest_categorical('word_max_features', [10000, 20000, 30000])
    
    word_vectorizer = TfidfVectorizer(
        analyzer='word', ngram_range=(1, 2), 
        min_df=w_min_df, max_df=w_max_df, max_features=w_max_feat
    )

    # 2. TF-IDF Caracteres
    c_min_df = trial.suggest_int('char_min_df', 1, 4)
    c_max_df = trial.suggest_float('char_max_df', 0.7, 1.0)
    c_max_feat = trial.suggest_categorical('char_max_features', [10000, 30000, 50000])
    
    char_vectorizer = TfidfVectorizer(
        analyzer='char', ngram_range=(3, 5), 
        min_df=c_min_df, max_df=c_max_df, max_features=c_max_feat
    )

    union = FeatureUnion([('word', word_vectorizer), ('char', char_vectorizer)])
    X_features = union.fit_transform(X_train) # Usamos tu texto enriquecido

    # 3. 🛑 EL GUARDARRAÍL: C nunca bajará de 0.1 🛑
    c_val = trial.suggest_float('C', 0.1, 10.0, log=True) 
    penalty = trial.suggest_categorical('penalty', ['l2', 'elasticnet'])
    
    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)
        solver = 'saga'
    else:
        l1_ratio = None
        solver = 'lbfgs'
        
    modelo = LogisticRegression(
        C=c_val, penalty=penalty, l1_ratio=l1_ratio, solver=solver,
        class_weight='balanced', max_iter=300, random_state=42
    )

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    score = cross_val_score(modelo, X_features, y_train, cv=skf, scoring='accuracy', n_jobs=-1)

    return score.mean()

print("Iniciando Optuna Seguro para el modelo con contexto...")
study_seguro = optuna.create_study(direction='maximize')
study_seguro.optimize(objective_contexto, n_trials=20) 

print("\n" + "="*50)
print("🏆 NUEVOS HIPERPARÁMETROS SEGUROS")
print("="*50)
print(f"Mejor Accuracy en entrenamiento: {study_seguro.best_value:.4f}")
for k, v in study_seguro.best_params.items():
    print(f"  - {k}: {v}")

[I 2026-05-18 21:21:35,033] A new study created in memory with name: no-name-2d283acb-d60d-4fb7-abf8-856b7c045696


Iniciando Optuna Seguro para el modelo con contexto...


[I 2026-05-18 21:24:03,308] Trial 0 finished with value: 0.6331832873623755 and parameters: {'word_min_df': 2, 'word_max_df': 0.9259522769044332, 'word_max_features': 30000, 'char_min_df': 3, 'char_max_df': 0.841504321975779, 'char_max_features': 50000, 'C': 6.889053176278311, 'penalty': 'elasticnet', 'l1_ratio': 0.03489914044499065}. Best is trial 0 with value: 0.6331832873623755.
[I 2026-05-18 21:24:05,534] Trial 1 finished with value: 0.6294957994018532 and parameters: {'word_min_df': 1, 'word_max_df': 0.881095043799051, 'word_max_features': 30000, 'char_min_df': 2, 'char_max_df': 0.8587881592141484, 'char_max_features': 30000, 'C': 3.204183177374614, 'penalty': 'l2'}. Best is trial 0 with value: 0.6331832873623755.
[I 2026-05-18 21:24:27,137] Trial 2 finished with value: 0.6254739023066965 and parameters: {'word_min_df': 3, 'word_max_df': 0.9437831312336394, 'word_max_features': 20000, 'char_min_df': 4, 'char_max_df': 0.8084352861120017, 'char_max_features': 30000, 'C': 3.554569431


🏆 NUEVOS HIPERPARÁMETROS SEGUROS
Mejor Accuracy en entrenamiento: 0.6349
  - word_min_df: 2
  - word_max_df: 0.812851388442456
  - word_max_features: 20000
  - char_min_df: 1
  - char_max_df: 0.9676046004459177
  - char_max_features: 50000
  - C: 0.6738749032967405
  - penalty: l2


🎯 Entrenamiento Final y Generación de Entrega
Ignoramos la variabilidad de Optuna y usamos exactamente los hiperparámetros que nos garantizaron el éxito anterior para generar el archivo de Kaggle.

In [3]:
# ==========================================
# 1. VECTORIZADORES CON HIPERPARÁMETROS ÓPTIMOS
# ==========================================
print("Vectorizando texto con los mejores parámetros de Optuna...")

mejor_word_seguro = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2), 
    min_df=2, max_df=0.7000457842048763, max_features=20000
)

mejor_char_seguro = TfidfVectorizer(
    analyzer='char', ngram_range=(3, 5), 
    min_df=1, max_df=0.9986253119965343, max_features=10000
)

mejor_union_segura = FeatureUnion([('word', mejor_word_seguro), ('char', mejor_char_seguro)])

# Asumimos que X_train, y_train y X_test siguen cargados de la celda del contexto
X_train_num_seguro = mejor_union_segura.fit_transform(X_train)
X_test_num_seguro = mejor_union_segura.transform(X_test)

# ==========================================
# 2. MODELO CON PARÁMETROS SEGUROS
# ==========================================
print("Entrenando el modelo final...")

modelo_final = LogisticRegression(
    C=0.3849932094413777, penalty='l2', solver='lbfgs',
    class_weight='balanced', max_iter=500, random_state=42
)

modelo_final.fit(X_train_num_seguro, y_train)
predicciones_finales = modelo_final.predict(X_test_num_seguro)

# ==========================================
# 3. CREAR ARCHIVO DE KAGGLE
# ==========================================
sample = pd.read_csv("../../data/sample_submission.csv") 
# Usamos el test original para sacar los IDs
test_original = pd.read_csv("../../data/test_nolabel.csv")

mis_predicciones = pd.DataFrame({'id': test_original['id'], 'label': predicciones_finales})

submission_final_optuna = sample[['id']].merge(mis_predicciones, on='id', how='left')
submission_final_optuna['label'] = submission_final_optuna['label'].fillna(1).astype(int)

nombre_archivo = 'submission_contexto_optuna1.csv'
submission_final_optuna.to_csv("../../data/" + nombre_archivo, index=False)

print(f"¡Listo! Todo generado correctamente. Sube el archivo '{nombre_archivo}' a Kaggle.")

Vectorizando texto con los mejores parámetros de Optuna...
Entrenando el modelo final...
¡Listo! Todo generado correctamente. Sube el archivo 'submission_contexto_optuna1.csv' a Kaggle.
